In [ ]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm

from loka.data import load_breakthrough_states
from loka.models import BreakthroughCNN

import os
import pandas as pd

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
BASE = "/home/axelbjarkar/T-404-LOKA/codebase"

X,Y = load_breakthrough_states(f"{BASE}/data/game_states/e.txt")

A forgetting event is when
\begin{equation}
\mathbf{1}_{(y^t_i = y_i)} = 1 > \mathbf{1}_{(y^{t+1}_i = y_i)} = 0
\end{equation}

This is calculated with
\begin{equation} C^{[t+1]}=C^{[t]}+\max\left(\mathbf{1}(Y=M^{[t]}(X))-\mathbf{1}(Y=M^{[t+1]}(X)),\mathbf{0}\right) \end{equation}

In [ ]:
start = 0.0001
end = 0.0002
steps = 2
batch_size = 512

splits = np.linspace(start, end, num=steps, endpoint=True)
mse_values = []
prc_values = []

X_t = torch.FloatTensor(X).to(device)
Y_t = torch.FloatTensor(Y).to(device)

epochs = 1
pbar = tqdm(total=steps * epochs, desc='training')

for seed, ratio in enumerate(splits):

    torch.manual_seed(SEED)  # same init weights each time
    model = BreakthroughCNN().to(device)

    # training setup
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    
    # convert data to tensors
    X_train_t = torch.FloatTensor(X_t).to(device)
    Y_train_t = torch.FloatTensor(Y_t).to(device)
    
    # training loop
    for epoch in range(epochs):
        perm = torch.randperm(len(X_train_t), device=device)  # new shuffle each epoch
        
        for i in range(0, len(X_train_t), batch_size):
            idx = perm[i:i + batch_size]
            xb, yb = X_train_t[idx], Y_train_t[idx]
    
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        pbar.update(1)

    pbar.set_postfix(model=f'{seed+1}/{steps}', ratio=f'{ratio:.2f}', loss=f'{loss.item():.4f}')
    
    # evaluate
    with torch.no_grad():
        test_preds = model(X_t)
        mse = loss_fn(test_preds, Y_t).item()
        mse_values.append(mse)
        prc_values.append(ratio)

pbar.close()